In [5]:
# Explore dataset

import pandas as pd
data = pd.read_csv('D:\\Real-Estate-AI-Assitant\\data\\cleaned_dataset.csv')

In [6]:
# Encode Categorical Variables

import sklearn.preprocessing as preprocessing
import sklearn.preprocessing as StandardScaler
label_encoder = preprocessing.LabelEncoder()
standard_scaler = StandardScaler.StandardScaler()

In [7]:
data['Lokasi']= label_encoder.fit_transform(data['Lokasi'])
data['Kota']= label_encoder.fit_transform(data['Kota'])


In [8]:
from sklearn.model_selection import train_test_split
Y = data['Harga']
X = data.drop('Harga', axis = 1)

X_train, X_test, y_train, y_test = train_test_split(X, Y, test_size=0.2, random_state=42, shuffle=True)

In [9]:
# Validation Set Creation
X_train, X_val, y_train, y_val = train_test_split(X_train,y_train, test_size=0.25, random_state=40, shuffle=True)  



In [10]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
import scipy.stats as stats

# Hyperparameter Tuning
rf = RandomForestRegressor(random_state=42)

# Define the hyperparameter grid
param_dist = {
    'n_estimators': stats.randint(50, 500),
    'max_depth': stats.randint(3, 20),
    'max_features' : ['sqrt', 'log2', None],
    'min_samples_split': stats.randint(2, 20),
    'min_samples_leaf': stats.randint(1, 20),
    'bootstrap': [True, False]
}



In [11]:
# Random Search CV
rf_random = RandomizedSearchCV(estimator = rf, param_distributions = param_dist, n_iter = 100, cv = 3, verbose=2, random_state=42, n_jobs = 1, scoring='neg_mean_squared_error', error_score='raise')
rf_random.fit(X_train, y_train)

print("Best Hyperparameters:", rf_random.best_params_)
tuned_model = rf_random.best_estimator_

Fitting 3 folds for each of 100 candidates, totalling 300 fits
[CV] END bootstrap=True, max_depth=17, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=171; total time=   0.3s
[CV] END bootstrap=True, max_depth=17, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=171; total time=   0.3s
[CV] END bootstrap=True, max_depth=17, max_features=None, min_samples_leaf=8, min_samples_split=8, n_estimators=171; total time=   0.3s
[CV] END bootstrap=True, max_depth=13, max_features=None, min_samples_leaf=4, min_samples_split=9, n_estimators=201; total time=   0.4s
[CV] END bootstrap=True, max_depth=13, max_features=None, min_samples_leaf=4, min_samples_split=9, n_estimators=201; total time=   0.4s
[CV] END bootstrap=True, max_depth=13, max_features=None, min_samples_leaf=4, min_samples_split=9, n_estimators=201; total time=   0.4s
[CV] END bootstrap=True, max_depth=4, max_features=log2, min_samples_leaf=6, min_samples_split=3, n_estimators=241; total

In [12]:
tuned_model.fit(X_train, y_train)
tuned_model.score(X_val, y_val)
print("Validation Set Score:", tuned_model.score(X_val, y_val))

#Test Data Evaluation
y_pred = tuned_model.predict(X_test)
from sklearn.metrics import mean_absolute_error, r2_score
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)
print("Test Set Mean Absolute Error:", mae)
print("Test Set R^2 Score:", r2)

Validation Set Score: 0.9991634014034981
Test Set Mean Absolute Error: 12196164.050145514
Test Set R^2 Score: 0.9981042426006076


In [13]:
# Check the scale of your predictions vs actual values
print("y_test range:", y_test.min(), "to", y_test.max())
print("y_pred range:", y_pred.min(), "to", y_pred.max())
print("Mean of y_test:", y_test.mean())
print("Mean of y_pred:", y_pred.mean())

# Look for extreme errors
errors = y_test - y_pred
print("Max error:", errors.max())
print("Min error:", errors.min())

y_test range: 300000000 to 9900000000
y_pred range: 304501856.43564355 to 9888714563.764069
Mean of y_test: 2961407703.83992
Mean of y_pred: 2961990792.72637
Max error: 1417924602.89985
Min error: -1725942828.9440665


In [14]:
import joblib
import os
from datetime import datetime

# Create a timestamp for versioning
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Path 
path = 'D:/Real-Estate-AI-Assitant/models/'

# Save the model
model_filename = os.path.join(path, f'rf_model_{timestamp}.pkl')
joblib.dump(tuned_model, model_filename)


# Save model metadata
metadata_filename = os.path.join(path, f'metadata_{timestamp}.pkl')
metadata = {
    'model_type': 'RandomForestRegressor',
    'mae': mae,
    'r2_score': r2,
    'train_date': timestamp,
    'feature_names': list(X_train.columns),  # if using DataFrame
    'best_params': tuned_model.get_params()
}

joblib.dump(metadata, metadata_filename)

print(f"Model saved as: {model_filename}")

Model saved as: D:/Real-Estate-AI-Assitant/models/rf_model_20260310_035428.pkl


In [15]:
# mlflow ui --port 5000
import mlflow

In [17]:
mlflow.set_experiment("Anomaly-Detection")
mlflow.set_tracking_uri("http://localhost:5000")

with mlflow.start_run(run_name='RealEstatePredictor'):
    mlflow.log_params(tuned_model.get_params())
    mlflow.log_metrics({"r2_score": r2})
    
    mlflow.sklearn.log_model(tuned_model, 'Random_Forest_Model')


2026/03/10 03:55:15 INFO mlflow.tracking.fluent: Experiment with name 'Anomaly-Detection' does not exist. Creating a new experiment.
2026/03/10 03:55:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/10 03:55:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


🏃 View run RealEstatePredictor at: http://localhost:5000/#/experiments/2/runs/c344e04b009745d6964dbb8ee1bfedb1
🧪 View experiment at: http://localhost:5000/#/experiments/2
